Configuração e Sessão Spark

In [ ]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
import zipfile
import shutil
import os
from pathlib import Path

spark = (
    SparkSession.builder
    .appName("sisvan") 
    .master("local[*]") 
    .config("spark.sql.shuffle.partitions", "16") 
    .config("spark.driver.memory", "4g")     
    .config("spark.executor.memory", "16g")  
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    .getOrCreate()
)


Definição de Caminhos e Variáveis

In [ ]:
DIRETORIO_ATUAL = Path.cwd()

PASTA_DADOS = DIRETORIO_ATUAL / "basesSISVAN"

PASTA_TEMP_EXTRACAO = PASTA_DADOS / "temp_extracao_csv"
PASTA_TEMP_STAGE = PASTA_DADOS / "temp_stage_parquet"
PASTA_TEMP_SPARK = PASTA_DADOS / "temp_spark_final"

NOME_ARQUIVO_FINAL = 'sisvanSjdr.sv'
CAMINHO_FINAL = PASTA_DADOS / NOME_ARQUIVO_FINAL

ANOS_ALVO = range(2017, 2024)
CODIGO_MUNICIPIO_ALVO = '316250' 


PASTA_TEMP_EXTRACAO.mkdir(parents=True, exist_ok=True)


O Loop de Processamento (ETL Incremental)

In [ ]:

for ano in ANOS_ALVO:
    zip_filename = f'sisvan_estado_nutricional_{ano}.zip'
    csv_filename = f'sisvan_estado_nutricional_{ano}.csv'
    
    zip_path = PASTA_DADOS / zip_filename
    
    if not zip_path.exists():
        print(f"AVISO: {zip_filename} não encontrado em {PASTA_DADOS}. Pulando...")
        continue
        
    print(f"[{ano}] 1. Extraindo arquivo do ZIP...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extract(csv_filename, path=PASTA_TEMP_EXTRACAO)
        
    csv_path = PASTA_TEMP_EXTRACAO / csv_filename
    
    print(f"[{ano}] 2. Lendo e Filtrando dados...")
    
    df_ano = spark.read.csv(
        str(csv_path), 
        header=True, 
        sep=';', 
        encoding='iso-8859-1'
    )
    
    df_sjdr = df_ano.filter(f.col('CO_MUNICIPIO_IBGE') == CODIGO_MUNICIPIO_ALVO)
    
    df_sjdr = df_sjdr.withColumn('ANO_ARQUIVO', f.lit(ano))

    # Salva o resultado parcial filtrado em Parquet
    (
        df_sjdr.write
        .mode('append')
        .parquet(str(PASTA_TEMP_STAGE))
    )
    
    # Libera memória
    df_ano.unpersist()
    del df_ano
    
    print(f"[{ano}] 3. Limpando CSV bruto...")
    os.remove(csv_path)



Unificação dos anos em um csv 

In [ ]:
if PASTA_TEMP_STAGE.exists() and any(PASTA_TEMP_STAGE.iterdir()):
    df_completo = spark.read.parquet(str(PASTA_TEMP_STAGE))
    
    (
        df_completo
        .coalesce(1) 
        .write
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "iso-8859-1")
        .mode('overwrite')
        .csv(str(PASTA_TEMP_SPARK))
    )
    
    # Renomear e Mover para o destino final
    part_file = next(PASTA_TEMP_SPARK.glob('part-*.csv'), None)
    
    if part_file:
        if CAMINHO_FINAL.exists():
            os.remove(CAMINHO_FINAL)
        
        shutil.move(str(part_file), str(CAMINHO_FINAL))
        print(f"SUCESSO! Arquivo salvo em: {CAMINHO_FINAL}")
    else:
        print("ERRO: O Spark não gerou o arquivo CSV.")
else:
    print("Nada para exportar (nenhum dado processado).")

Limpeza final

In [ ]:
if PASTA_TEMP_EXTRACAO.exists(): shutil.rmtree(PASTA_TEMP_EXTRACAO)
if PASTA_TEMP_STAGE.exists(): shutil.rmtree(PASTA_TEMP_STAGE)
if PASTA_TEMP_SPARK.exists(): shutil.rmtree(PASTA_TEMP_SPARK)

spark.stop()